In [ ]:
from collections.abc import Iterable
import os

import certifi
import urllib3
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEndpoint,
    HuggingFaceEndpointEmbeddings,
 )
from langgraph.graph import MessagesState
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Suppress only the warning emitted when verify=False is used as a fallback.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CA_BUNDLE = certifi.where()

# Requires HUGGINGFACEHUB_API_TOKEN in the notebook environment.
HF_EMBEDDING_MODEL = os.getenv(
    "HF_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
 )
HF_LLM_REPO_ID = os.getenv(
    "HF_LLM_REPO_ID", "mistralai/Mistral-7B-Instruct-v0.3"
 )

List the ECB blog pages that will be downloaded and indexed for question answering.

In [ ]:
urls = [
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260506~586ab63f08.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260421~3c1c5a08ee.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260413~78ef6fe470.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260409~cc951a0d29.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260407~dfa96b8bfc.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260402~d9be74e490.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260331~af8055c801.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260330~0370e4586a.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260327~51b0640c39.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260321~0df0ef78e7.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260312~5dfa697fdd.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260310~b738caf5b4.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260309~3a4841b028.en.html",
    "https://www.ecb.europa.eu/press/blog/date/2026/html/ecb.blog20260304~d9e34fc95f.en.html",
]

Create a loader that retries with different SSL verification settings when a site certificate causes download failures.

In [ ]:
def load_with_ssl_fallback(url: str):
    attempts = [
        ("system-ca", True),
        ("certifi", CA_BUNDLE),
        ("insecure", False),
    ]

    last_error = None
    for mode, verify_value in attempts:
        try:
            loader = WebBaseLoader(
                url,
                requests_kwargs={"verify": verify_value, "timeout": 30},
            )
            if mode == "insecure":
                print(f"Using insecure SSL fallback for {url}")
            return loader.load()
        except Exception as exc:
            last_error = exc
            if mode != "insecure":
                print(f"SSL mode '{mode}' failed for {url}. Trying next fallback.")

    raise RuntimeError(f"All SSL strategies failed for {url}") from last_error

Fetch each page and collect all returned documents into one list for later processing.

In [ ]:
docs = []
for url in urls:
    docs.extend(load_with_ssl_fallback(url))

Flatten nested results so the retriever pipeline always works with standard LangChain Document objects.

In [ ]:
def normalize_documents(items):
    normalized = []
    for item in items:
        if isinstance(item, Document):
            normalized.append(item)
        elif isinstance(item, Iterable) and not isinstance(item, (str, bytes, dict)):
            for inner in item:
                if isinstance(inner, Document):
                    normalized.append(inner)
    return normalized

Break the articles into smaller passages so retrieval works on focused sections instead of full pages.

In [ ]:
docs_list = normalize_documents(docs)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50,
 )
doc_splits = text_splitter.split_documents(docs_list)

Create Hugging Face embeddings, store the chunks in memory, and expose a retriever for semantic search.

In [ ]:
embedding_model = HuggingFaceEndpointEmbeddings(model=HF_EMBEDDING_MODEL)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=embedding_model
 )
retriever = vectorstore.as_retriever()

Keep document lookup in one function so the response step can reuse it cleanly.

In [ ]:
def retrieve_blog_posts(query: str) -> str:
    """Search and return information from ECB blog posts."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

Connect to the Hugging Face inference endpoint that will generate the final natural-language response.

In [ ]:
hf_llm = HuggingFaceEndpoint(
    repo_id=HF_LLM_REPO_ID,
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
 )
response_model = ChatHuggingFace(llm=hf_llm)

Retrieve relevant context for the latest question and ask the model to answer using that evidence.

In [ ]:
def generate_query_or_respond(state: MessagesState):
    """Retrieve relevant ECB posts and answer the latest user question."""
    user_message = state["messages"][-1].content
    context = retrieve_blog_posts(user_message)

    response = response_model.invoke(
        [
            (
                "system",
                "You answer questions about ECB blog posts. Use the retrieved context when it is relevant and say when the context is insufficient.",
            ),
            (
                "human",
                f"Question: {user_message}\n\nRetrieved context:\n{context}",
            ),
        ]
    )
    return {"messages": [response]}

# Exercise: Metadata-Guided Blog Post Analysis with LangChain

In this notebook, you built a retrieval pipeline over ECB blog posts using Hugging Face embeddings and a Hugging Face inference model. The next step is to make the workflow more structured by first inspecting the metadata of each blog post, classifying its subject with an LLM, and then using that classification to decide which follow-up prompts should be run.

The goal of this exercise is to turn the current notebook into a metadata-guided analysis pipeline that combines classification and prompt routing.

### Task:
Use the blog post metadata, such as title, publication date, and URL, as input to an LLM that assigns a subject label to each post.
In particular, you should:
- Extract or organize the available metadata for each blog post.
- Use an LLM to classify each post into a subject category such as inflation, monetary policy, banking, financial stability, digital euro, or another meaningful set of labels.
- Define follow-up prompts that depend on the assigned category.
- Run category-specific prompts after classification, for example a summary prompt for one category and a comparison or risk-analysis prompt for another.
- Check whether the classification is consistent and whether the routed prompts produce more relevant outputs than a single generic prompt.

### Objective:
Build a workflow in which the LLM first uses metadata to understand what a post is about, and then uses that classification to trigger more targeted prompts and more useful downstream analysis.

### Optional bonus (advanced)
- Compare metadata-only classification with classification that also uses retrieved text snippets.
- Allow a post to receive multiple subject labels when appropriate.
- Create a small evaluation table showing the metadata, predicted label, follow-up prompt used, and quality of the final output.